In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")



Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas7_KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

# Menyembunyikan log INFO agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/10 17:21:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/10 17:21:30 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
# Membaca data langsung dari HDFS
df_transaksi_kn = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang_kn = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu_kn = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

In [7]:
df_gabungan_kn = df_transaksi_kn.join(df_cabang_kn, on="cabang_id", how="inner") \
                                .join(df_menu_kn, on="menu_id", how="inner")

df_gabungan_kn = df_gabungan_kn.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan_kn.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [8]:
ringkasan_menu_cabang = df_gabungan_kn.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

top2_menu_cabang = ringkasan_menu_cabang.withColumn(
    "peringkat", row_number().over(window_cabang)
).filter(col("peringkat") <= 2)

top2_menu_cabang.orderBy("nama_cabang", "peringkat").show()

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



In [9]:
df_gabungan_kn.createOrReplaceTempView("kopi_nusantara")

kontribusi_barista = spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM kopi_nusantara
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
''')
kontribusi_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [10]:
path_output = "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"

# Simpan data dengan mode overwrite dan partisi
df_gabungan_kn.write.mode("overwrite").partitionBy("kategori_menu").parquet(path_output)

# Verifikasi pembacaan kembali
df_verifikasi = spark.read.parquet(path_output)
print("Verifikasi Load Berhasil. Jumlah baris data:", df_verifikasi.count())

Verifikasi Load Berhasil. Jumlah baris data: 4000


In [11]:
ringkasan_eksekutif_kn = df_verifikasi.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan_triwulan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan_triwulan").desc())

ringkasan_eksekutif_kn.show()

[Stage 28:>                                                         (0 + 2) / 2]

+--------------------+----------+--------------+------------------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan_triwulan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+------------------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|                90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|                87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|                84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+------------------------+----------------+-------------------------+



Berdasarkan hasil analisis data transaksi triwulanan menggunakan Apache Spark, kita dapat menarik wawasan strategis mengenai kinerja setiap cabang Kopi Nusantara. Dari tabel ringkasan eksekutif, terlihat jelas bahwa Kopi Nusantara Semarang yang dipimpin oleh kepala barista Nadia merupakan cabang dengan kinerja terbaik. Cabang ini memimpin dengan membukukan total penjualan tertinggi mencapai Rp 90.542.000 serta volume transaksi tertinggi sebanyak 1.393 pesanan. Pencapaian ini menunjukkan bahwa operasional dan daya tarik menu di cabang Semarang berjalan dengan sangat maksimal.   Di sisi lain, Kopi Nusantara Yogyakarta mencatatkan kinerja terlemah pada triwulan ini dengan total penjualan terendah (Rp 84.645.000), meskipun secara volume transaksi sedikit lebih baik dibandingkan Magelang. Untuk mengatasi kelemahan ini, manajemen disarankan untuk mengoptimalkan menu-menu terlaris (Top-1 dan Top-2 dari analisis sebelumnya) yang disukai pelanggan di cabang Yogyakarta. Menu andalan tersebut sebaiknya dipromosikan lebih gencar melalui strategi paket bundling atau promosi silang (cross-selling) dengan produk-produk yang penjualannya cenderung melambat (slow-moving). Strategi promosi gabungan ini diharapkan mampu mendongkrak rata-rata nilai transaksi (average order value) secara efektif, sehingga total pendapatan cabang Yogyakarta dapat segera meningkat tanpa membutuhkan biaya pemasaran yang besar, sekaligus memaksimalkan peran kepala barista Reza dalam mengeksekusi layanan yang berfokus pada penjualan . 